# PHOBERT MULTI-TASK V3.1: PURE SENTIMENT & MULTI-LABEL ASPECT

## 1. CHUẨN BỊ MÔI TRƯỜNG COLAB VÀ IMPORT TỪ SRC

In [ ]:
#Kiem tra moi truong Colab, giai nen src_and_data.zip neu co va cai thu vien con thieu
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('src/modeling.py') and os.path.exists('/content/src_and_data.zip'):
    import zipfile
    with zipfile.ZipFile('/content/src_and_data.zip', 'r') as zf: zf.extractall('.')
for pkg in ['transformers', 'accelerate', 'pyvi', 'scipy']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
#Bat buoc ton tai package src/ va du lieu sach *_clean.csv (khong viet ham duplicate inline)
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path: sys.path.insert(0, ROOT)
assert os.path.exists(os.path.join(ROOT, 'src', 'modeling.py')), 'Thieu src/modeling.py! Hay clone repo hoac upload/unzip src_and_data.zip len Colab.'
assert os.path.exists(os.path.join(ROOT, 'data', 'processed', 'train_clean.csv')), 'Thieu data/processed/train_clean.csv!'

In [ ]:
import json, zipfile, joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [ ]:
#Import Single Source of Truth tu package src/
from src.labels import ASPECT_GROUPS
from src.text_cleaner import clean_text, segment_words
from src.modeling import (
    MAX_LEN, PII_TOKENS, PhoBertMultiTask, TemperatureScaler, ViSFDDataset,
    dynamic_collate_fn, init_semantic_pii_embeddings, multilabel_to_primary_aspect,
    split_dev_three_way, tune_calib_thresholds,
)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')
SAVE_DIR = os.path.join(MODEL_DIR, 'phobert_multitask')
os.makedirs(SAVE_DIR, exist_ok=True)
print('in_colab: ', IN_COLAB, '| device: ', device)
if IN_COLAB and not torch.cuda.is_available(): print('CANH BAO: Hay chon Runtime -> Change runtime type -> T4 GPU!')

## 2. NẠP DỮ LIỆU SẠCH VÀ CHIA TẬP DEV 3 PHẦN ĐỘC LẬP

In [ ]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_clean.csv'))
asp_cols = ['asp_0', 'asp_1', 'asp_2', 'asp_3', 'asp_4']

In [ ]:
df_train.shape, df_dev.shape, df_test.shape

In [ ]:
df_train[['sentence_seg', 'sentiment', 'aspect_group'] + asp_cols].head(3)

In [ ]:
#Chia Dev thanh 3 tap tach biet: Eyeball (250), Calib (300), Blackbox (562)
df_eyeball, df_calib, df_blackbox = split_dev_three_way(df_dev, random_state=42)
df_eyeball.shape, df_calib.shape, df_blackbox.shape

In [ ]:
pd.DataFrame({
    'eyeball': df_eyeball['sentiment'].value_counts(),
    'calib': df_calib['sentiment'].value_counts(),
    'blackbox': df_blackbox['sentiment'].value_counts(),
}).sort_index()

## 3. TOKENIZER, KHỞI TẠO EMBEDDING PII VÀ DYNAMIC DATALOADER

In [ ]:
#Nap tokenizer va khoi tao vector ngu nghia cho 5 special token PII (khong de random)
MODEL_NAME = 'vinai/phobert-base-v2'
tok_src = SAVE_DIR if os.path.exists(os.path.join(SAVE_DIR, 'vocab.txt')) else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tok_src)
DROPOUT = 0.3
if torch.cuda.is_available():
    model = PhoBertMultiTask(model_name_or_dir=MODEL_NAME, dropout=DROPOUT)
else:
    cfg = AutoConfig.from_pretrained(SAVE_DIR if os.path.exists(os.path.join(SAVE_DIR, 'config.json')) else MODEL_NAME)
    model = PhoBertMultiTask(config=cfg, dropout=DROPOUT)
init_semantic_pii_embeddings(model.encoder, tokenizer)
model = model.to(device)

In [ ]:
#Kiem tra nhanh van ban giu nguyen chu hoa/thuong (cased) va token PII khong bi vo
tempt = segment_words(clean_text('Màn hình đẹp, liên hệ 0901234567 đơn ORD-2024-001'))
tempt, tokenizer.tokenize(tempt)

In [ ]:
#Tao Dataset giu cased sentence_seg, khong padding truoc de dynamic_collate_fn xu ly theo batch (MAX_LEN=256)
train_ds = ViSFDDataset(df_train['sentence_seg'].fillna(''), tokenizer, df_train['sentiment'], df_train[asp_cols].values, MAX_LEN)
eye_ds = ViSFDDataset(df_eyeball['sentence_seg'].fillna(''), tokenizer, df_eyeball['sentiment'], df_eyeball[asp_cols].values, MAX_LEN)
cal_ds = ViSFDDataset(df_calib['sentence_seg'].fillna(''), tokenizer, df_calib['sentiment'], df_calib[asp_cols].values, MAX_LEN)
blk_ds = ViSFDDataset(df_blackbox['sentence_seg'].fillna(''), tokenizer, df_blackbox['sentiment'], df_blackbox[asp_cols].values, MAX_LEN)
dev_ds = ViSFDDataset(df_dev['sentence_seg'].fillna(''), tokenizer, df_dev['sentiment'], df_dev[asp_cols].values, MAX_LEN)
test_ds = ViSFDDataset(df_test['sentence_seg'].fillna(''), tokenizer, df_test['sentiment'], df_test[asp_cols].values, MAX_LEN)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
BATCH_SIZE = 32 if torch.cuda.is_available() else 16
collate = lambda b: dynamic_collate_fn(b, pad_id=tokenizer.pad_token_id)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
eye_loader = DataLoader(eye_ds, batch_size=64, shuffle=False, collate_fn=collate)
cal_loader = DataLoader(cal_ds, batch_size=64, shuffle=False, collate_fn=collate)
blk_loader = DataLoader(blk_ds, batch_size=64, shuffle=False, collate_fn=collate)
dev_loader = DataLoader(dev_ds, batch_size=64, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate)

In [ ]:
b0 = next(iter(train_loader))
b0['input_ids'].shape, b0['attention_mask'].shape, b0['sentiment'].shape, b0['aspects'].shape

## 4. HÀM LOSS ĐA NHIỆM VÀ CẤU HÌNH HUẤN LUYỆN

In [ ]:
#Trong so can bang lop cho Sentiment (CrossEntropy) va Multi-Label Aspect (BCEWithLogitsLoss)
s_cnt = df_train['sentiment'].value_counts().sort_index().values
w_s = torch.tensor(np.sqrt(len(df_train) / (3.0 * s_cnt)), dtype=torch.float).to(device)
y_tr_asp = df_train[asp_cols].values
pos_cnt = y_tr_asp.sum(axis=0)
pos_w_a = torch.tensor(np.sqrt((len(df_train) - pos_cnt) / np.maximum(pos_cnt, 1)), dtype=torch.float).to(device)
loss_sent_fn = nn.CrossEntropyLoss(weight=w_s, label_smoothing=0.05)
loss_asp_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w_a)
print('w_s: ', w_s.cpu().numpy().round(3), '| pos_w_a: ', pos_w_a.cpu().numpy().round(3))

In [ ]:
LR = 2e-5
WEIGHT_DECAY = 0.02
EPOCHS = 6
PATIENCE = 2
weights_path = os.path.join(SAVE_DIR, 'phobert_multitask.pt')
history_path = os.path.join(SAVE_DIR, 'training_history.json')
calib_path = os.path.join(SAVE_DIR, 'calibration.json')

In [ ]:
def collect_logits(m, loader):
    m.eval()
    ls_all, la_all = [], []
    with torch.no_grad():
        for b in loader:
            ls, la = m(b['input_ids'].to(device), b['attention_mask'].to(device))
            ls_all.append(ls.cpu().numpy()); la_all.append(la.cpu().numpy())
    return np.vstack(ls_all), np.vstack(la_all)

## 5. HUẤN LUYỆN TRÊN GPU VỚI REAL EARLY STOPPING

In [ ]:
#Tach weight_decay=0.0 cho bias va LayerNorm; Early Stopping theo blackbox_mean_f1 voi patience=2
if torch.cuda.is_available():
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    opt_groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(opt_groups, lr=LR)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    best_blk_f1, wait, history = -1.0, 0, []
    for ep in range(EPOCHS):
        model.train()
        tot_loss, tr_ps, tr_pa, tr_ys, tr_ya = 0.0, [], [], [], []
        for b in train_loader:
            optimizer.zero_grad()
            ls, la = model(b['input_ids'].to(device), b['attention_mask'].to(device))
            loss = loss_sent_fn(ls, b['sentiment'].to(device)) + 0.9 * loss_asp_fn(la, b['aspects'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); scheduler.step()
            tot_loss += loss.item()
            tr_ps.extend(ls.argmax(dim=-1).cpu().numpy()); tr_pa.append((la.detach().cpu().numpy() >= 0.0).astype(int))
            tr_ys.extend(b['sentiment'].numpy()); tr_ya.append(b['aspects'].numpy().astype(int))
        tr_sf1 = f1_score(tr_ys, tr_ps, average='macro')
        tr_af1 = f1_score(np.vstack(tr_ya), np.vstack(tr_pa), average='macro', zero_division=0)
        blk_ls, blk_la = collect_logits(model, blk_loader)
        blk_sf1 = f1_score(df_blackbox['sentiment'], blk_ls.argmax(axis=-1), average='macro')
        blk_af1 = f1_score(df_blackbox[asp_cols].values, (blk_la >= 0.0).astype(int), average='macro', zero_division=0)
        blk_mf1 = (blk_sf1 + blk_af1) / 2.0
        if blk_mf1 > best_blk_f1:
            best_blk_f1, wait, st = blk_mf1, 0, 'best_saved'
            torch.save(model.state_dict(), weights_path)
            tokenizer.save_pretrained(SAVE_DIR)
            model.encoder.config.save_pretrained(SAVE_DIR)
        else:
            wait += 1
            st = 'early_stop' if wait >= PATIENCE else f'patience_{wait}'
        history.append({'epoch': ep+1, 'loss': round(tot_loss/len(train_loader), 4), 'train_sent_f1': round(tr_sf1, 4), 'train_asp_f1': round(tr_af1, 4), 'blackbox_sent_f1': round(blk_sf1, 4), 'blackbox_multilabel_asp_f1': round(blk_af1, 4), 'blackbox_mean_f1': round(blk_mf1, 4), 'status': st})
        with open(history_path, 'w', encoding='utf-8') as f: json.dump(history, f, indent=2, ensure_ascii=False)
        print('epoch:', ep+1, '| blackbox_mean_f1:', round(blk_mf1, 4), '| status:', st)
        if wait >= PATIENCE:
            print('Dung som (Early Stopping) tai epoch:', ep+1)
            break
else:
    print('Dang tren CPU Local: Hay chay Notebook 04 tren Google Colab T4 GPU de huan luyen v3.1')

In [ ]:
if os.path.exists(history_path):
    pd.read_json(history_path)

## 6. HIỆU CHỈNH TEMPERATURE SCALING VÀ NGƯỠNG TRÊN DF_CALIB

In [ ]:
#Nap lai checkpoint tot nhat va hieu chinh nhiet do sau khi huan luyen xong (chi dung df_calib)
if not os.path.exists(weights_path): raise FileNotFoundError('Chua co phobert_multitask.pt!')
cfg_best = AutoConfig.from_pretrained(SAVE_DIR)
model = PhoBertMultiTask(config=cfg_best, dropout=DROPOUT)
if model.encoder.get_input_embeddings().weight.shape[0] != len(tokenizer): model.encoder.resize_token_embeddings(len(tokenizer))
model.load_state_dict(torch.load(weights_path, map_location=device))
model = model.to(device)

In [ ]:
cal_ls, cal_la = collect_logits(model, cal_loader)
scaler = TemperatureScaler().fit(cal_ls, cal_la, df_calib['sentiment'].values, df_calib[asp_cols].values)
cal_sp, cal_ap = scaler.predict_probs(cal_ls, cal_la)
calib_info = {**scaler.to_dict(), **tune_calib_thresholds(cal_sp, cal_ap, df_calib['sentiment'].values, df_calib[asp_cols].values)}
with open(calib_path, 'w', encoding='utf-8') as f: json.dump(calib_info, f, indent=2, ensure_ascii=False)
calib_info

## 7. PHÂN TÍCH LỖI TRÊN DF_EYEBALL SO VỚI BASELINE

In [ ]:
asp_th = calib_info['aspect_threshold']
conf_th = calib_info['low_conf_threshold']
eye_ls, eye_la = collect_logits(model, eye_loader)
eye_sp, eye_ap = scaler.predict_probs(eye_ls, eye_la)
eye_s_pred = eye_sp.argmax(axis=-1)
eye_a_bin = (eye_ap >= asp_th).astype(int)
eye_a_prim = multilabel_to_primary_aspect(eye_ap, asp_th)
print('eyeball sent f1:', round(f1_score(df_eyeball['sentiment'], eye_s_pred, average='macro'), 4), '| multilabel asp f1:', round(f1_score(df_eyeball[asp_cols].values, eye_a_bin, average='macro', zero_division=0), 4))

In [ ]:
#So sanh so cau Baseline doan sai tren df_eyeball duoc PhoBERT sua dung
base_pipe_p, vec_p, svc_p = os.path.join(MODEL_DIR, 'baseline_sentiment.pkl'), os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'), os.path.join(MODEL_DIR, 'svc_sentiment.pkl')
base_eye = joblib.load(base_pipe_p).predict(df_eyeball['sentence_seg'].fillna('')) if os.path.exists(base_pipe_p) else (joblib.load(svc_p).predict(joblib.load(vec_p).transform(df_eyeball['sentence_seg'].fillna(''))) if os.path.exists(vec_p) and os.path.exists(svc_p) else None)
if base_eye is not None:
    err_mask = (df_eyeball['sentiment'].values != base_eye)
    fixed_mask = (df_eyeball['sentiment'].values == eye_s_pred) & err_mask
    print('baseline sai tren eyeball:', int(err_mask.sum()), '| phobert sua dung:', int(fixed_mask.sum()), f'({fixed_mask.sum()/max(err_mask.sum(),1)*100:.1f}%)')

## 8. ĐÁNH GIÁ NIÊM PHONG TRÊN BLACKBOX DEV VÀ TEST

In [ ]:
blk_ls, blk_la = collect_logits(model, blk_loader)
blk_sp, blk_ap = scaler.predict_probs(blk_ls, blk_la)
blk_s_pred, blk_a_bin, blk_a_prim = blk_sp.argmax(axis=-1), (blk_ap >= asp_th).astype(int), multilabel_to_primary_aspect(blk_ap, asp_th)
blk_eff = blk_a_bin.copy()
blk_no_pos = blk_eff.sum(axis=1) == 0
blk_eff[blk_no_pos, blk_a_prim[blk_no_pos]] = 1
blk_hit = float(((blk_eff & df_blackbox[asp_cols].values).sum(axis=1) > 0).mean())
print('blackbox sent f1:', round(f1_score(df_blackbox['sentiment'], blk_s_pred, average='macro'), 4), '| multilabel asp f1:', round(f1_score(df_blackbox[asp_cols].values, blk_a_bin, average='macro', zero_division=0), 4), '| primary asp f1:', round(f1_score(df_blackbox['aspect_group'], blk_a_prim, average='macro'), 4), '| hit-any:', round(blk_hit, 4))

In [ ]:
test_ls, test_la = collect_logits(model, test_loader)
test_sp, test_ap = scaler.predict_probs(test_ls, test_la)
test_s_pred, test_a_bin, test_a_prim = test_sp.argmax(axis=-1), (test_ap >= asp_th).astype(int), multilabel_to_primary_aspect(test_ap, asp_th)
test_eff = test_a_bin.copy()
test_no_pos = test_eff.sum(axis=1) == 0
test_eff[test_no_pos, test_a_prim[test_no_pos]] = 1
test_hit = float(((test_eff & df_test[asp_cols].values).sum(axis=1) > 0).mean())
sent_f1_test, sent_acc_test = f1_score(df_test['sentiment'], test_s_pred, average='macro'), accuracy_score(df_test['sentiment'], test_s_pred)
asp_ml_f1_test = f1_score(df_test[asp_cols].values, test_a_bin, average='macro', zero_division=0)
asp_f1_test, asp_acc_test = f1_score(df_test['aspect_group'], test_a_prim, average='macro'), accuracy_score(df_test['aspect_group'], test_a_prim)
print('test sent f1:', round(sent_f1_test, 4), '| acc:', round(sent_acc_test, 4))
print('test multilabel asp f1:', round(asp_ml_f1_test, 4), '| primary asp f1:', round(asp_f1_test, 4), '| hit-any:', round(test_hit, 4))

In [ ]:
print(classification_report(df_test['sentiment'], test_s_pred, target_names=['Tiêu cực', 'Trung lập', 'Tích cực'], digits=4))

In [ ]:
print(classification_report(df_test['aspect_group'], test_a_prim, target_names=ASPECT_GROUPS, digits=4))

## 9. XUẤT DỰ ĐOÁN, METADATA VÀ ĐÓNG GÓI PHOBERT_ECOMMERCE_OUTPUTS_V3.ZIP

In [ ]:
#Suy luan full Dev va xuat ca dev_pred_phobert.csv lan test_pred_phobert.csv kem needs_review da hieu chinh
dev_ls, dev_la = collect_logits(model, dev_loader)
dev_sp, dev_ap = scaler.predict_probs(dev_ls, dev_la)
dev_s_pred, dev_a_bin, dev_a_prim = dev_sp.argmax(axis=-1), (dev_ap >= asp_th).astype(int), multilabel_to_primary_aspect(dev_ap, asp_th)
dev_pred_path = os.path.join(PROC_DIR, 'dev_pred_phobert.csv')
test_pred_path = os.path.join(PROC_DIR, 'test_pred_phobert.csv')
for df_src, sp_a, ap_a, s_p, a_b, a_p, out_p in [(df_dev, dev_sp, dev_ap, dev_s_pred, dev_a_bin, dev_a_prim, dev_pred_path), (df_test, test_sp, test_ap, test_s_pred, test_a_bin, test_a_prim, test_pred_path)]:
    df_o = df_src.copy()
    df_o['pred_sentiment'], df_o['pred_aspect'] = s_p, a_p
    for i in range(5): df_o[f'pred_asp_{i}'] = a_b[:, i]
    df_o['sentiment_conf'], df_o['aspect_conf'] = np.round(sp_a.max(axis=-1), 4), np.round(ap_a.max(axis=-1), 4)
    contra = ((df_o['n_star'] <= 2) & (df_o['pred_sentiment'] == 2)) | ((df_o['n_star'] >= 4) & (df_o['pred_sentiment'] == 0))
    low_c = (df_o['sentiment_conf'] < conf_th) | (df_o['aspect_conf'] < conf_th)
    df_o['needs_review'] = (low_c | contra | df_o['is_no_content'].astype(bool)).astype(bool)
    df_o.to_csv(out_p, index=False, encoding='utf-8-sig')

In [ ]:
df_te_check = pd.read_csv(test_pred_path)
df_te_check['needs_review'].value_counts()

In [ ]:
df_te_check[df_te_check['needs_review']][['sentence_clean', 'n_star', 'pred_sentiment', 'pred_aspect', 'sentiment_conf', 'aspect_conf']].head(4)

In [ ]:
#Dong bo chi so Baseline va PhoBERT v3.1 vao model_metadata.json
meta = {'dataset': f'UIT-ViSFD E-Commerce ({len(df_train)} train / {len(df_eyeball)} eyeball / {len(df_calib)} calib / {len(df_blackbox)} blackbox / {len(df_test)} test)', 'calibration': calib_info, 'phobert_multitask': {'backbone': MODEL_NAME, 'sentiment_macro_f1': round(float(sent_f1_test), 4), 'sentiment_accuracy': round(float(sent_acc_test), 4), 'aspect_multilabel_macro_f1': round(float(asp_ml_f1_test), 4), 'aspect_macro_f1': round(float(asp_f1_test), 4), 'aspect_accuracy': round(float(asp_acc_test), 4), 'hit_any_true_group': round(float(test_hit), 4)}}
base_csv = os.path.join(PROC_DIR, 'test_pred_baseline.csv')
if os.path.exists(base_csv):
    dfb = pd.read_csv(base_csv)
    meta['baseline_tfidf'] = {'sentiment_macro_f1': round(float(f1_score(dfb['sentiment'], dfb['pred_sentiment'], average='macro')), 4), 'sentiment_accuracy': round(float(accuracy_score(dfb['sentiment'], dfb['pred_sentiment'])), 4), 'aspect_macro_f1': round(float(f1_score(dfb['aspect_group'], dfb['pred_aspect'], average='macro')), 4), 'aspect_accuracy': round(float(accuracy_score(dfb['aspect_group'], dfb['pred_aspect'])), 4)}
    if all(f'pred_asp_{i}' in dfb.columns for i in range(5)): meta['baseline_tfidf']['aspect_multilabel_macro_f1'] = round(float(f1_score(dfb[asp_cols].values, dfb[[f'pred_asp_{i}' for i in range(5)]].values, average='macro', zero_division=0)), 4)
with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w', encoding='utf-8') as f: json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))

In [ ]:
#Dong goi toan bo artifact PhoBERT v3.1 + tokenizer + calib + dev/test csv vao phobert_ecommerce_outputs_v3.zip
zip_out = os.path.join(ROOT, 'phobert_ecommerce_outputs_v3.zip')
tok_files = ['added_tokens.json', 'special_tokens_map.json', 'vocab.txt', 'bpe.codes', 'tokenizer_config.json']
pack = [(weights_path, 'phobert_multitask.pt'), (os.path.join(SAVE_DIR, 'config.json'), 'config.json'), (history_path, 'training_history.json'), (calib_path, 'calibration.json'), (dev_pred_path, 'dev_pred_phobert.csv'), (test_pred_path, 'test_pred_phobert.csv'), (os.path.join(MODEL_DIR, 'model_metadata.json'), 'model_metadata.json')]
pack += [(os.path.join(SAVE_DIR, fn), fn) for fn in tok_files]
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p, arc in pack:
        if os.path.exists(p): zf.write(p, arc)
print('da dong goi v3:', zip_out)